# Phase 3 : Préparation des Features pour le Modèle ML
Ce notebook prépare les données pour la prédiction de l'ISRB (Indice de Sécurité). L'objectif est d'agréger les données du GDELT au niveau mensuel et départemental, de calculer l'ISRB courant, de créer les features et de décaler la cible pour prédire le mois suivant.

> NOTE
> - Filtrage géographique ciblé (exclusion du bruit national `BN` et `BN00`).
> - Création d'une grille temporelle complète pour combler les trous et éviter les décalages de cibles erronés.

In [7]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

pd.set_option('display.max_columns', None)

os.makedirs('../data/processed', exist_ok=True)

## Chargement et filtrage géographique
Nous filtrons spécifiquement les données pour exclure `BN` (Bénin en général) et `BN00`, afin de conserver uniquement les événements précisément localisés dans un département.

In [2]:
# Charger les données
df = pd.read_csv('../data/clean/gdelt_benin_clean.csv')

print(f"Taille des données en entrée: {df.shape}")
df.head()

Taille des données en entrée: (22778, 64)


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL,year,month,week
0,1248558842,2025-06-09,202506,2025,2025.4356,BEN,COTONOU,BEN,NaN,NaN,...,BN,NaN,9.5,2.25,BN,20250609020000,https://english.news.cn/20250609/833e6be5b34f4...,2025,6,24
1,1248561281,2025-06-09,202506,2025,2025.4356,NGA,BENIN CITY,NGA,NaN,NaN,...,BN,NaN,9.5,2.25,BN,20250609023000,https://punchng.com/ndlea-intercepts-drugs-dis...,2025,6,24
2,1248560915,2025-06-09,202506,2025,2025.4356,BEN,BENIN,BEN,NaN,NaN,...,BN,NaN,9.5,2.25,BN,20250609023000,https://punchng.com/ndlea-intercepts-drugs-dis...,2025,6,24
3,1248556983,2025-06-09,202506,2025,2025.4356,AFR,AFRICA,AFR,NaN,NaN,...,BN,NaN,9.5,2.25,BN,20250609013000,https://www.eurasiareview.com/09062025-us-afri...,2025,6,24
4,1248557466,2025-06-09,202506,2025,2025.4356,USAGOV,US OFFICIAL,USA,NaN,NaN,...,BN,NaN,9.5,2.25,BN,20250609013000,https://www.eurasiareview.com/09062025-us-afri...,2025,6,24


In [12]:
df['ActionGeo_ADM1Code'].unique()

<ArrowStringArray>
[  'BN', 'BN09', 'BN07', 'BN00', 'BN12', 'BN16', 'BN18', 'BN10', 'BN13',
 'BN08', 'BN15', 'BN11', 'BN14']
Length: 13, dtype: str

In [13]:
# Filtrer le bruit national
df['ActionGeo_ADM1Code'] = df['ActionGeo_ADM1Code'].fillna('Unknown')
to_exclude = ['BN', 'BN00']
df_filtered = df[~df['ActionGeo_ADM1Code'].isin(to_exclude)].copy()

print(f"Événements conservés (localisés) : {len(df_filtered)}")

Événements conservés (localisés) : 1415


## 2. Calcul des métriques par événement avant agrégation
Nous identifions les valeurs négatives pour le GoldsteinScale et l'AvgTone.

In [14]:
df_filtered['GoldsteinScale'] = pd.to_numeric(df_filtered['GoldsteinScale'], errors='coerce')
df_filtered['AvgTone'] = pd.to_numeric(df_filtered['AvgTone'], errors='coerce')

df_filtered['Goldstein_Neg'] = df_filtered['GoldsteinScale'].apply(lambda x: x if x < 0 else np.nan)
df_filtered['AvgTone_Neg'] = df_filtered['AvgTone'].apply(lambda x: x if x < 0 else np.nan)
df_filtered['Is_QuadClass4'] = (df_filtered['QuadClass'] == 4).astype(int)

mil_cop = ['MIL', 'COP']
df_filtered['Has_MIL_COP'] = (
    df_filtered['Actor1Type1Code'].isin(mil_cop) |
    df_filtered['Actor1Type2Code'].isin(mil_cop) |
    df_filtered['Actor2Type1Code'].isin(mil_cop) |
    df_filtered['Actor2Type2Code'].isin(mil_cop)
).astype(int)

## 3. Agrégation Mensuelle par Département
On groupe par `MonthYear` et `ActionGeo_ADM1Code`.

In [15]:
agg_df = df_filtered.groupby(['MonthYear', 'ActionGeo_ADM1Code']).agg(
    Total_Events=('GLOBALEVENTID', 'count'),
    Goldstein_Neg_Mean=('Goldstein_Neg', 'mean'),
    AvgTone_Neg_Mean=('AvgTone_Neg', 'mean'),
    QuadClass4_Count=('Is_QuadClass4', 'sum'),
    MIL_COP_Count=('Has_MIL_COP', 'sum')
).reset_index()

agg_df['Pct_QuadClass4'] = (agg_df['QuadClass4_Count'] / agg_df['Total_Events']) * 100
agg_df['Goldstein_Neg_Mean'] = agg_df['Goldstein_Neg_Mean'].fillna(0)
agg_df['AvgTone_Neg_Mean'] = agg_df['AvgTone_Neg_Mean'].fillna(0)

## 4. Grille temporelle (Remplissage des mois manquants)
Indispensable pour que le `shift(-1)` décale réellement d'un seul mois, même en l'absence d'événements.

In [16]:
all_months = sorted(agg_df['MonthYear'].unique())
all_depts = agg_df['ActionGeo_ADM1Code'].unique()

grid = pd.MultiIndex.from_product([all_months, all_depts], names=['MonthYear', 'ActionGeo_ADM1Code']).to_frame(index=False)
agg_df_full = pd.merge(grid, agg_df, on=['MonthYear', 'ActionGeo_ADM1Code'], how='left')

for col in ['Total_Events', 'Goldstein_Neg_Mean', 'AvgTone_Neg_Mean', 'QuadClass4_Count', 'MIL_COP_Count', 'Pct_QuadClass4']:
    agg_df_full[col] = agg_df_full[col].fillna(0)

## 5. Calcul de l'ISRB et Target Shiftée

In [18]:
agg_df_full['Goldstein_Neg_Norm'] = agg_df_full[['Goldstein_Neg_Mean']].abs() / 10.0
agg_df_full['AvgTone_Neg_Norm'] = agg_df_full[['AvgTone_Neg_Mean']].abs() / 10.0
agg_df_full['Pct_QuadClass4_Norm'] = agg_df_full['Pct_QuadClass4'] / 100.0

agg_df_full['ISRB'] = (agg_df_full['Pct_QuadClass4_Norm'] * 0.40) + \
                      (agg_df_full['Goldstein_Neg_Norm'] * 0.35) + \
                      (agg_df_full['AvgTone_Neg_Norm'] * 0.25)

# Shift de la target par département
agg_df_full = agg_df_full.sort_values(['ActionGeo_ADM1Code', 'MonthYear']).reset_index(drop=True)
agg_df_full['Target_ISRB_NextMonth'] = agg_df_full.groupby('ActionGeo_ADM1Code')['ISRB'].shift(-1)

final_df = agg_df_full.dropna(subset=['Target_ISRB_NextMonth']).copy()
print(f"Dimensions après création de la target : {final_df.shape}")

Dimensions après création de la target : (132, 13)


## 6. Encodage du département (OneHotEncoding)

In [19]:
ohe = OneHotEncoder(sparse_output=False, drop='first')
dept_encoded = ohe.fit_transform(final_df[['ActionGeo_ADM1Code']])
dept_cols = [f"Dept_{c}" for c in ohe.categories_[0][1:]]

dept_encoded_df = pd.DataFrame(dept_encoded, columns=dept_cols, index=final_df.index)
final_df = pd.concat([final_df, dept_encoded_df], axis=1)
final_df.head()

,MonthYear,ActionGeo_ADM1Code,Total_Events,Goldstein_Neg_Mean,AvgTone_Neg_Mean,QuadClass4_Count,MIL_COP_Count,Pct_QuadClass4,Goldstein_Neg_Norm,AvgTone_Neg_Norm,Pct_QuadClass4_Norm,ISRB,Target_ISRB_NextMonth,Dept_BN08,Dept_BN09,Dept_BN10,Dept_BN11,Dept_BN12,Dept_BN13,Dept_BN14,Dept_BN15,Dept_BN16,Dept_BN18
0,202504,BN07,7.0,-6.800000,-9.446170,3.0,0.0,42.857143,0.680000,0.944617,0.428571,0.645583,0.570078,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,202505,BN07,17.0,-9.533333,-3.809378,6.0,4.0,35.294118,0.953333,0.380938,0.352941,0.570078,0.042017,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,202506,BN07,1.0,0.000000,-1.680672,0.0,0.0,0.000000,0.000000,0.168067,0.000000,0.042017,0.792481,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,202507,BN07,6.0,-9.000000,-11.099253,3.0,1.0,50.000000,0.900000,1.109925,0.500000,0.792481,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,202508,BN07,4.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.645776,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 7. Exportation

In [20]:
output_path = '../data/processed/features_ml.csv'
final_df.to_csv(output_path, index=False)
print(f"Données exportées avec succès vers {output_path} (Dimensions : {final_df.shape})")

Données exportées avec succès vers ../data/processed/features_ml.csv (Dimensions : (132, 23))


## Analyses extra (par curiosité)

In [38]:
# Sélection des colonnes numériques pour la corrélation
# On exclut les colonnes de type string ou ID
from sklearn.preprocessing import LabelEncoder

agg_df_full['dept'] = LabelEncoder().fit_transform(agg_df_full['ActionGeo_ADM1Code'])
numeric_cols = agg_df_full.select_dtypes(include=[np.number]).columns
corr_matrix = agg_df_full[numeric_cols].corr()

# Corrélation spécifique avec la cible
target_corr = corr_matrix['Target_ISRB_NextMonth'].sort_values(ascending=False)

print("=== Corrélations avec la Target (ISRB_NextMonth) ===")
print(target_corr)

# Analyse des features clés
print("\n=== Analyse des features clés ===")
key_features = ['ISRB', 'Total_Events', 'Pct_QuadClass4', 'Goldstein_Neg_Mean', 'AvgTone_Neg_Mean', 'MIL_COP_Count']
for feat in key_features:
    if feat in target_corr:
        print(f"Corrélation {feat} <-> Target : {target_corr[feat]:.4f}")

=== Corrélations avec la Target (ISRB_NextMonth) ===
Target_ISRB_NextMonth    1.000000
QuadClass4_Count         0.241662
AvgTone_Neg_Norm         0.237254
Pct_QuadClass4_Norm      0.174568
Pct_QuadClass4           0.174568
ISRB                     0.173545
Total_Events             0.125946
MonthYear                0.102670
Goldstein_Neg_Norm       0.102307
MIL_COP_Count            0.088818
Goldstein_Neg_Mean      -0.102307
AvgTone_Neg_Mean        -0.237254
dept                    -0.344544
Name: Target_ISRB_NextMonth, dtype: float64

=== Analyse des features clés ===
Corrélation ISRB <-> Target : 0.1735
Corrélation Total_Events <-> Target : 0.1259
Corrélation Pct_QuadClass4 <-> Target : 0.1746
Corrélation Goldstein_Neg_Mean <-> Target : -0.1023
Corrélation AvgTone_Neg_Mean <-> Target : -0.2373
Corrélation MIL_COP_Count <-> Target : 0.0888


In [22]:
np.corrcoef(final_df['ISRB'], final_df['Target_ISRB_NextMonth'])

array([[1.        , 0.17354525],
       [0.17354525, 1.        ]])